In [0]:
%run /Workspace/Users/wenjie.chen@bayer.com/OE/00_Helper_Notebooks/00_connection

In [0]:
df = spark.read \
    .format("delta") \
    .load("/Volumes/ph-com-ai-us-prod-catalog-2553213839599380/oe_journey/oe_volume_space/notebook_output/final_mdb")

In [0]:
# removing census features
cols_to_drop = ['ZIP_CODE','TERR_ID','MONTH_ID','ACCT_NAME','MAJOR_CLASS_OF_TRADE']

feature_columns = df.columns

census_features = [col for col in feature_columns if col.startswith("ACS_PCT_")]
print(len(census_features))

from collections import OrderedDict

to_drop = list(OrderedDict.fromkeys((census_features + cols_to_drop)))

df = df.drop(*to_drop)
print("Shape :", df.count(), len(df.columns))

In [0]:
from pyspark.sql.types import StringType, DoubleType, LongType, DateType, IntegerType, FloatType, ArrayType, DecimalType

categoricalcols = [f.name for f in df.schema.fields if isinstance(f.dataType, StringType)]
DoubleTypescols = [f.name for f in df.schema.fields if isinstance(f.dataType, DoubleType)]
LongTypecols = [f.name for f in df.schema.fields if isinstance(f.dataType, LongType)]
DateTypecols = [f.name for f in df.schema.fields if isinstance(f.dataType, DateType)]
IntTypecols = [f.name for f in df.schema.fields if isinstance(f.dataType, IntegerType)]
FloatTypecols = [f.name for f in df.schema.fields if isinstance(f.dataType, FloatType)]
ArrayTypecols = [f.name for f in df.schema.fields if isinstance(f.dataType, ArrayType)]
DecimalTypecols = [f.name for f in df.schema.fields if isinstance(f.dataType, DecimalType)]

NumericTypecols = [
    f.name for f in df.schema.fields
    if not isinstance(f.dataType, StringType)
    and not isinstance(f.dataType, DateType)
    and not isinstance(f.dataType, ArrayType)
]

# All columns except array and datetype
Allcols = [
    f.name for f in df.schema.fields
    if not isinstance(f.dataType, DateType)
    and not isinstance(f.dataType, ArrayType)
]

print("Length of Categorical features:", len(categoricalcols))
print("Length of Double Type features:", len(DoubleTypescols))
print("Length of Long Type features:", len(LongTypecols))
print("Length of Date Type features:", len(DateTypecols))
print("Length of Int Type features:", len(IntTypecols))
print("Length of Float Type features:", len(FloatTypecols))
print("Length of Array Type features:", len(ArrayTypecols))
print("Length of Decimal Type features:", len(DecimalTypecols))
print("Length of Numerical Type features:", len(NumericTypecols))
print("Length of All features:", len(Allcols))

In [0]:
print('HCP_COHORT_ID' in df.columns)

In [0]:
key_cols = ['PROVIDER_ID','INSTANCE']

df_wo_cat = df.drop(*categoricalcols)
df_wo_cat = df_wo_cat.drop(*key_cols)
print("Shape :", df_wo_cat.count(), len(df_wo_cat.columns))

In [0]:
  print('ACCT_NAME' in df.columns)

In [0]:
from pyspark.sql.functions import stddev, col
from pyspark.sql.types import NumericType

# Step 1.1: Remove low variance
numeric_cols = [
    f.name for f in df_wo_cat.schema.fields
    if isinstance(f.dataType, NumericType) and f.name != "TOTAL_NBRX_C3"]

stddevs = df_wo_cat.select([stddev(col(c)).alias(c) for c in numeric_cols])
stddev_dict = stddevs.collect()[0].asDict()

non_zero_std_cols = [c for c, std in stddev_dict.items() if std is not None and std > 1e-6]
print("Number of columns with non-zero standard deviation:", len(non_zero_std_cols))

df_wo_cat = df_wo_cat.select(non_zero_std_cols + ['TOTAL_NBRX_C3'])
print("Shape :", df_wo_cat.count(), len(df_wo_cat.columns))

In [0]:
from pyspark.sql.functions import count, when
import pandas as pd

# Step 1.2: Checking Null Rate - No feature is removed from this step
row_count = df_wo_cat.count()
null_ratios = df_wo_cat.select([(count(when(col(c).isNull(), c)) / row_count).alias(c) for c in df_wo_cat.columns])
nulls = null_ratios.collect()[0].asDict()

pd.DataFrame(list(nulls.items()), columns=['column_name', 'null_ratio']).sort_values(by='null_ratio', ascending=False)

In [0]:
df_non_zero = df_wo_cat.filter(col('TOTAL_NBRX_C3') > 0)
df_non_zero = df_non_zero.toPandas()

X_nz = df_non_zero.drop('TOTAL_NBRX_C3', axis=1)
y_nz = df_non_zero['TOTAL_NBRX_C3']

mi_scores = mutual_info_regression(X_nz, y_nz)
mi_series = pd.Series(mi_scores, index=X_nz.columns)

top_features = mi_series.sort_values(ascending=False).head(5000).index.tolist()

df_wo_cat = df_wo_cat.select(*top_features, 'TOTAL_NBRX_C3')

print("Shape :", df_wo_cat.count(), len(df_wo_cat.columns))